# Show the Inflow along a Transect into the 79NG Fjord

Notebook by Markus Reinert (IOW, 2024–2026, https://orcid.org/0000-0002-3761-8029).

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import xarray as xr
import cmocean.cm as cmo
import matplotlib.pyplot as plt

from tools.visualization import extend_cbar, cm

## Define the tools

In [ ]:
def get_edges(coordinate_array):
    dx = np.diff(coordinate_array)
    return np.concatenate((
        [coordinate_array[0] - dx[0]/2],
        coordinate_array[:-1] + dx/2,
        [coordinate_array[-1] + dx[-1]/2],
    ))

In [ ]:
def get_mesh_coords(section):
    """Get x- and z-coordinate edges to draw section with pcolormesh.

    For the z-coordinate edges outside the section domain, a constant
    extrapolation is used.  A linear interpolation (which is the
    default of pcolormesh) can give incorrect results if bathymetry or
    ice topography have high slopes at a section boundary.
    """
    # Check the dimensions of section
    assert len(section.dims) == 2, "section must have exactly 2 dimensions"
    assert "level" in section.dims, 'section must have dimension "level"'
    # Get the name of the second dimension
    dim = [dim for dim in section.dims if dim != "level"][0]
    # Make sure that masked values have been removed
    if "mask" in section:
        assert all(section.mask) and not any(np.isnan(section.mask)), "section must not contain masked values"
    # Get the x-coordinate edges
    x = section[dim].data
    xe = get_edges(x)
    # Get the z-coordinate edges
    ze = np.zeros((section.level.size + 1, xe.size))
    for i in range(len(ze)):
        ze[i] = np.interp(
            xe, x, -section.bathymetry + (section.h.isel(level=slice(i)).sum("level") if i else 0)
        )
    return xe, ze

In [ ]:
def get_bulk_values(ds, layer, variables=[], name="layer"):
    data = {}
    data["h"] = ds.h.where(layer).sum("level")
    data["h"].attrs.update(long_name=f"{name} thickness", units=ds.h.units)
    for var in variables:
        data[var] = (ds[var] * ds.h).where(layer).sum("level") / data["h"]
        data[var].attrs.update(long_name=f"{name} {ds[var].long_name}", units=ds[var].units)
    return data

In [ ]:
def do_plume_analysis(ds, layer, velocity, ambient_density):
    g = 9.81
    name = "plume"
    data = get_bulk_values(ds, layer, [velocity, "rho"], name)
    # Change the name of the velocity data to "u" if needed
    if velocity != "u":
        data["u"] = data[velocity]
        del data[velocity]
    data["rho_a"] = ambient_density * xr.ones_like(data["rho"])
    data["rho_a"].attrs.update(long_name="ambient density", units=data["rho"].units)
    data["drho"] = data["rho"] - data["rho_a"]
    data["drho"].attrs.update(long_name="density difference", units=data["rho"].units)
    data["b"] = g * data["drho"] / data["rho_a"]
    data["b"].attrs.update(long_name=f"{name} buoyancy", units="m/s$^2$")
    data["Fr"] = data["u"] / np.sqrt(data["b"] * data["h"])
    data["Fr"].attrs.update(long_name="Froude number")
    return data

## Load the model result

In [ ]:
ds = xr.open_dataset("output/out_mean_compressed.nc").set_coords("zc")

# Convert single time value from coordinate to attribute
assert ds.time.size == 1
time_value = ds.time.data[0]
ds = ds.squeeze(dim="time", drop=True)
ds.attrs["time"] = time_value

# Add (missing) geometry variables to the dataset
with xr.open_dataset("output/out_geometry_2d.nc") as geo:
    for var in geo.variables:
        if var in ds:
            print(var, "exists in dataset and geometry.")
            # Check that the data matches between dataset and geometry
            xr.testing.assert_equal(ds[var], geo[var])
        else:
            ds[var] = geo[var]

# Compute the mask
ds["mask"] = ds.bathymetry.notnull()
ds.mask.attrs = {"long_name": "ocean mask"}

# Check grid spacing
dlat = 1 / 240
dlon = 3 / 120
assert np.allclose(np.diff(ds.latc), dlat)
assert np.allclose(np.diff(ds.lonc), dlon)

# Use this aspect ratio in lat–lon plots for squared grid cells; since dx and dy
# are approximately equal, this gives an approximate equal-area map.
grille_carree = dlon / dlat

# Update the attributes for better labels with xarray
ds.temp.attrs.update({"long_name": "Temperature", "units": "°C"})
ds.salt.attrs.update({"long_name": "Salinity", "units": "g kg$^{-1}$"})

ds

## Load the inflow path

This path has been constructed in the notebook `Inflow_analysis_path.ipynb` based on model topography and results.

In [ ]:
path = xr.open_dataset("data/path_inflow.nc")
# Put the start point in flow direction at kilometer 0
path["dist"] = ("dist", (path.dist - path.dist.max()).data, path.dist.attrs)
path.plot.scatter(x="lonc", y="latc", hue="dist")
path

## Analyze inflow along path

### Extract data along path

In [ ]:
inflow = ds.sel(path)
inflow

### Compute mesh coordinates

In [ ]:
xe, ZE = get_mesh_coords(inflow)

### Compute the direction of the inflow

We compute the direction of the inflow path using grid indices, which is close to the direction on a Cartesian grid, since the grid cells are close to squares.
At first, we compute the direction on the middle points between two path points, then we interpolate them to the path points.

In [ ]:
# Compute the step size in index coordinates along the path
di = np.diff(path.lonc / dlon)
dj = np.diff(path.latc / dlat)
# Check that di and dj are essentially integers
assert np.allclose(di.round(), di)
assert np.allclose(dj.round(), dj)
# Duplicate the first and last value
di = np.concatenate((di[:1], di, di[-1:]))
dj = np.concatenate((dj[:1], dj, dj[-1:]))
# Compute the step size on the path points
di_center = (di[1:] + di[:-1]) / 2
dj_center = (dj[1:] + dj[:-1]) / 2
# Flip the sign to point in inflow direction
di_center = -di_center
dj_center = -dj_center
# Compute the absolute value
dij_abs = np.sqrt(di_center**2 + dj_center**2)

fig, ax = plt.subplots(dpi=180)
ax.set_title("Direction along the path")
ax.plot(path.lonc, path.latc, "r.-")
plt.quiver(path.lonc, path.latc, di_center, dj_center, dij_abs, zorder=3)
plt.quiver(path.lonc, path.latc, dj_center, -di_center, dij_abs, zorder=3)
plt.colorbar(label="normalized length of the hypotenuse")
ax.set_aspect(grille_carree)

### Rotate velocity to inflow direction

In [ ]:
inflow["vel3d_along"] = (inflow.velx3d * di_center + inflow.vely3d * dj_center) / dij_abs
inflow.vel3d_along.attrs = {"long_name": "velocity along the path", "units": "m s$^{-1}$"}
inflow["vel3d_across"] = (inflow.velx3d * dj_center - inflow.vely3d * di_center) / dij_abs
inflow.vel3d_across.attrs = {"long_name": "velocity in across-path direction", "units": "m s$^{-1}$"}

### Compute the mixing

In [ ]:
assert (inflow.nuh.isel(level=-1) == 0).all()
assert (inflow.nuh.isel(level=slice(-1)) > 0).all()
mix_S = 2 * inflow.nuh.isel(level=slice(-1)) * (inflow.salt.diff("level", label="lower") / inflow.zc.diff("level", label="lower"))**2
mix_T = 2 * inflow.nuh.isel(level=slice(-1)) * (inflow.temp.diff("level", label="lower") / inflow.zc.diff("level", label="lower"))**2

print(f"Salinity    mixing range: {mix_S.min():.2} to {mix_S.max():.2}")
print(f"Temperature mixing range: {mix_T.min():.2} to {mix_T.max():.2}")

### Analyse the plume

Using the method by Schaffer et al. (2020):
This method fixes two isopycnals with densities $\rho_0 < \rho_1$.
We consider all water with density $\rho \geq \rho_1$ as the bottom layer and all water with density between $\rho_0$ and $\rho_1$ as the middle layer.
The bulk values of the layers are given by the vertical averages over the layers.
We consider the bottom layer as the plume and the middle layer as stagnant ambient water.

#### Select isopycnals of interest

In [ ]:
rho0 = 1027.2
rho1 = 1027.5

#### Select and show the 2 layers

In [ ]:
is_bottom_layer = inflow.rho >= rho1
is_middle_layer = (inflow.rho >= rho0) & (inflow.rho < rho1)
is_middle_layer.plot(y="zc", add_colorbar=False)
is_bottom_layer.where(is_bottom_layer | is_middle_layer).plot(y="zc", vmin=-1, cmap="viridis", add_colorbar=False)

#### Compute bulk values

In [ ]:
ambient = get_bulk_values(inflow, is_middle_layer, ["rho", "vel3d_along"])
plume = do_plume_analysis(inflow, is_bottom_layer, "vel3d_along", ambient["rho"])
# Note that `vel3d_along` has been renamed `u` for the plume,
# as it is given as the third argument to `do_plume_analysis`.
plume["temp"] = get_bulk_values(inflow, is_bottom_layer, ["temp"])["temp"]

print("Bulk quantitites of ambient water:", ", ".join(ambient))
print("Bulk quantitites of plume water:  ", ", ".join(plume))

plume_z = plume["h"] - inflow.bathymetry

### Compute bulk values for an alternative plume definition

To show the sensitivity of the bulk values to the chosen isopycnal.

In [ ]:
rho_alt = 1027.45
is_bottom_layer_alt = inflow.rho >= rho_alt
is_middle_layer_alt = (inflow.rho >= rho0) & (inflow.rho < rho_alt)

ambient_alt_rho = get_bulk_values(inflow, is_middle_layer_alt, ["rho"])["rho"]
plume_alt = do_plume_analysis(inflow, is_bottom_layer_alt, "vel3d_along", ambient_alt_rho)
plume_alt["temp"] = get_bulk_values(inflow, is_bottom_layer_alt, ["temp"])["temp"]

color_alt = "#aaaaaa"

## Prepare the figure

In [ ]:
plt.rcParams["axes.titlesize"] = "medium"
plt.rcParams["axes.labelsize"] = "small"
plt.rcParams["xtick.labelsize"] = "small"
plt.rcParams["ytick.labelsize"] = "small"

## Create the figure

In [ ]:
fig, axs = plt.subplots(3, 3, sharex=True, constrained_layout=True, figsize=(18*cm, 14*cm), dpi=200)
fig.suptitle("Dense bottom plume flowing into the 79° North Glacier fjord", weight="bold")

for ax, var in zip(axs.flat, ["temp", "salt", "rho", "mix", "vel3d_along"]):
    if var == "mix":
        data = mix_T
        name = r"Vertical mixing, $2K_v(\partial_zT)^2$"
        unit = "(°C)$^2$ s$^{-1}$"
        Z = ZE[:-1]
    elif var == "rho":
        data = inflow[var] - 1000
        name = r"Density, $\rho - 1000$"
        unit = "kg m$^{-3}$"
        Z = ZE
    else:
        data = inflow[var]
        name = data.long_name.capitalize()
        unit = data.units
        Z = ZE
    if var == "vel3d_along":
        # Flip the sign of the data and the colorbar ticks
        # to have positive velocity in blue on the left
        data = -data
        name = name.replace("the path", "transect")
    dmin = data.min()
    dmax = data.max()
    print(f"{dmin:9.6f} to {dmax:9.6f} is the range of {name}")
    if var == "temp":
        kwargs = {"cmap": cmo.thermal, "vmin": 0.2, "vmax": 2.0}
        ticks = np.linspace(kwargs["vmin"], kwargs["vmax"], 4)
    elif var == "salt":
        kwargs = {"cmap": cmo.haline, "vmin": 33.9, "vmax": 34.6}
        ticks = np.linspace(kwargs["vmin"], kwargs["vmax"], 3)
    elif var == "rho":
        kwargs = {"cmap": cmo.dense, "vmin": 27.2, "vmax": 27.6}
        ticks = np.linspace(kwargs["vmin"], kwargs["vmax"], 3)
    elif var == "mix":
        kwargs = {"cmap": cmo.amp, "vmin": 1e-8, "vmax": 1e-5, "norm": "log"}
        ticks = np.logspace(np.log10(kwargs["vmin"]), np.log10(kwargs["vmax"]), 4)
    elif var == "vel3d_along":
        vmax = 0.5  # alternative: max(-dmin, dmax)
        kwargs = {"cmap": cmo.balance, "vmin": -vmax, "vmax": vmax}
        ticks = np.linspace(-0.5, 0.5, 3)
        # Annotate the flow direction
        ax.annotate("", (-9, -270), (-6.5, -270), arrowprops={"arrowstyle": "fancy", "facecolor": "none", "edgecolor": "0.2", "linewidth": 0.75})
    im = ax.pcolormesh(xe, Z, data, **kwargs)
    # Add a colorbar in the sill
    cax = ax.inset_axes([-11.8, -360, 9, 12], transform=ax.transData)
    cbar = fig.colorbar(im, cax=cax, orientation="horizontal", **extend_cbar(dmin, dmax, *im.get_clim()), ticks=ticks)
    cbar.set_label(unit, labelpad=2)
    if var == "vel3d_along":
        # Flip the sign of the colorbar ticks according to the data (see above)
        cbar.set_ticklabels(f"${t:.1f}$" for t in ticks[::-1])
    # Show the density contours used for the plume analysis
    cs = (inflow.rho - 1000).plot.contour(ax=ax, y="zc", levels=[rho0 - 1000, rho1 - 1000], colors="r", linewidths=0.75, linestyles="--")
    if var == "rho":
        ax.clabel(cs)
    # Show the density contour of the alternative plume definition
    inflow.rho.plot.contour(ax=ax, y="zc", levels=[rho_alt], colors=color_alt, linewidths=0.75, linestyles="--")
    # Style the plot
    ax.set(title=name, ylim=(-460, -150), facecolor="0.8")
    if var == "temp" or var == "mix":
        ax.set_ylabel("depth in m")
        ax.yaxis.set_major_formatter(lambda x, pos: f"{-round(x)}")
    else:
        ax.set_ylabel("")
        ax.yaxis.set_major_formatter("")

for ax, var in zip(axs.flat[5:], ["h", "temp", "u", "Fr"]):
    plume[var].plot(ax=ax, label=fr"plume is $\rho \geq {rho1 - 1000:.1f}$")
    plume_alt[var].plot(ax=ax, label=fr"plume is $\rho \geq {rho_alt - 1000:.2f}$", color=color_alt)
    ax.set_ylabel("")
    ax.grid(axis="x", ls=":")

ax = axs[2, 2]
ax.set(title="Froude number, Fr", yticks=[0, 1, 2])
ax.axhline(1, color="k", lw=1, ls="--")
ax.annotate(
    "Fr > 1\nplume becomes\nsupercritical",
    ((plume["Fr"] <= 1).idxmax(), 1), (-8.5, 1.2),
    arrowprops={"arrowstyle": "->", "relpos": (0, 0.5)},
    size="small",
)

axs[2, 0].set_yticks([1.5, 1.75, 2.0])
axs[1, 2].legend(loc="upper left", fontsize="small")

for ax, title in zip(axs.flat[5:], ["Plume thickness in m", "Plume temperature in °C", "Plume velocity in m s$^{-1}$"]):
    ax.set_title(title)

for ax in [axs[1, 2], axs[2, 1], axs[2, 2]]:
    ax.set_ylim(0)

for letter, ax in zip("abcdefghi", axs.flat):
    ax.set(xlabel="distance in km" if letter in "ghi" else "", xticks=[-15, -10, -5, 0])
    ax.xaxis.set_major_formatter(lambda x, pos: f"{-x}")
    ax.set_title(f"({letter}) {ax.get_title()}")

fig.savefig("figures/fig_6_inflow_transect.png", dpi=600)